# 09a — Create Discover Domains, Subdomains & Pages

Organises the demo's gold tables, metric views, SQL functions, and Genie spaces
into a **business-aligned hierarchy** on the Databricks **Discover** page:

* **Domain:** SAP Purchasing (parent)
* **Subdomain 1:** Sourcing & Supplier Risk
* **Subdomain 2:** Purchase Order Analytics

Also creates **5 governed Pages** (business-term definitions) that Genie One reads
as part of its ontology — grounding natural-language answers in official definitions.

> **Pre-requisites:** gold tables, semantic layer, and Genie spaces must exist
> (steps 1–6 in `00_run_all`).
>
> **Permission:** requires `MANAGE DISCOVERY` on the account.

In [0]:
%run ../00_config

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.tags import TagPolicy
import json, time

w = WorkspaceClient()
NS = CFG.namespace

# ---- Thin REST helper with proper headers ----
def _do(method, path, body=None, params=None):
    """Call the Databricks REST API via the SDK's authenticated client."""
    headers = {"Accept": "application/json", "Content-Type": "application/json"}
    if w.config.workspace_id:
        headers["X-Databricks-Workspace-Id"] = w.config.workspace_id
    return w.api_client.do(method, path, body=body, query=params, headers=headers)

print(f"Workspace: {w.config.host}")
print(f"Namespace: {NS}")

## 1) Create governed tags
Domains are built on **governed tags**. We create a parent tag and two sub-tags
following the `{parent}/{child}` naming convention that Databricks uses for
subdomains.

In [0]:
# ---- Governed-tag definitions ----
# Parent domain tag
PARENT_TAG = "SAP_Purchasing"
# Subdomain tags (must start with parent prefix + "/")
SUB_TAG_1 = f"{PARENT_TAG}/Sourcing_Supplier_Risk"
SUB_TAG_2 = f"{PARENT_TAG}/Purchase_Order_Analytics"


GOVERNED_TAGS_OK = True  # Track whether governed tags were created

def ensure_governed_tag(tag_key: str, description: str = ""):
    """Create a governed tag (tag policy) if it does not already exist.
    Uses the SDK TagPoliciesAPI (POST /api/2.1/tag-policies).
    Falls back gracefully if account tag-policy limit is reached.
    """
    global GOVERNED_TAGS_OK
    # First check if it already exists
    try:
        for p in w.tag_policies.list_tag_policies():
            if p.tag_key == tag_key:
                print(f"  • Governed tag already exists: {tag_key} (id={p.id})")
                return True
    except Exception:
        pass
    # Try to create
    try:
        result = w.tag_policies.create_tag_policy(
            TagPolicy(tag_key=tag_key, description=description)
        )
        print(f"  ✓ Created governed tag: {tag_key} (id={result.id})")
        return True
    except Exception as e:
        if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
            print(f"  • Governed tag already exists: {tag_key}")
            return True
        elif "maximum" in str(e).lower() and "tag policies" in str(e).lower():
            GOVERNED_TAGS_OK = False
            print(f"  ⚠ Account tag-policy limit reached — cannot create governed tag: {tag_key}")
            print(f"    The account has reached the max number of governed tags (tag policies).")
            print(f"    An account admin must delete unused tag policies to free capacity.")
            print(f"    Falling back to regular (non-governed) tags for asset tagging.")
            return False
        else:
            raise


tag1_ok = ensure_governed_tag(PARENT_TAG, "SAP Purchasing domain — parent tag for the Discover hierarchy.")
tag2_ok = ensure_governed_tag(SUB_TAG_1, "Sourcing & Supplier Risk subdomain.")
tag3_ok = ensure_governed_tag(SUB_TAG_2, "Purchase Order Analytics subdomain.")

if not (tag1_ok and tag2_ok and tag3_ok):
    print()
    print("=" * 72)
    print("⚠  ACCOUNT TAG-POLICY LIMIT REACHED")
    print("=" * 72)
    print("This account has hit the maximum number of governed tag policies.")
    print("Governed tags could NOT be created for this demo.")
    print()
    print("Impact:")
    print("  • Asset tagging (Step 3)   → proceeds with regular tags  ✓")
    print("  • Domain creation (Step 2) → SKIPPED (requires governed tags)")
    print("  • Pages creation (Step 4)  → SKIPPED (requires domains)")
    print("  • Domain publish (Step 5)  → SKIPPED (requires domains)")
    print()
    print("To resolve:")
    print("  An account admin must delete unused tag policies to free")
    print("  capacity, then re-run this notebook.")
    print("=" * 72)
else:
    print("\n✓ All governed tags ready.")

## 2) Create the domain and subdomains
The Domains REST API creates entries on the **Discover** page. A domain maps 1:1
to a governed tag; subdomains use the `{parentTag}/{name}` convention.

In [0]:
# ---- Discover the correct domains API path ----
# The Domains API lives at /api/2.0/domains (NOT under /unity-catalog/).
DOMAINS_API = None
for candidate in ["/api/2.0/domains", "/api/2.1/domains",
                  "/api/2.0/unity-catalog/domains", "/api/2.1/unity-catalog/domains"]:
    try:
        _do("GET", candidate)
        DOMAINS_API = candidate
        break
    except Exception as e:
        if "Not Found" not in str(e) and "No API found" not in str(e):
            DOMAINS_API = candidate
            break

if DOMAINS_API:
    print(f"Domains API found: {DOMAINS_API}")
else:
    print("\u26a0\ufe0f  Domains REST API not available on this workspace.")
    print("   The 'Domains and Discover Page' preview must be enabled.")
    print("   Governed tags + asset tagging will still proceed.")


def find_domain(tag_key: str):
    """Return the domain/subdomain ID for a given tag_key, or None."""
    if not DOMAINS_API:
        return None
    try:
        resp = _do("GET", DOMAINS_API)
        for d in resp.get("domains", []):
            if d.get("tagName") == tag_key:
                return d.get("domainId")
    except Exception:
        pass
    return None


def create_or_get_domain(tag_key: str, subtitle: str, description: str,
                         icon_name: str = "SHOPPING_CART", icon_color: str = "#1B5E20",
                         parent_domain_id: str = None):
    """Idempotent: create a domain/subdomain, or return existing ID."""
    if not GOVERNED_TAGS_OK:
        print(f"  • Skipped domain (governed tags unavailable): {tag_key}")
        return None
    if not DOMAINS_API:
        print(f"  • Skipped domain creation (API unavailable): {tag_key}")
        return None
    existing = find_domain(tag_key)
    if existing:
        print(f"  • Domain already exists for tag '{tag_key}' → {existing}")
        return existing
    body = {
        "tag_key": tag_key,
        "subtitle": subtitle,
        "description": description,
        "icon": {"name": icon_name, "color": icon_color},
    }
    if parent_domain_id:
        body["parent_domain_id"] = parent_domain_id
    resp = _do("POST", DOMAINS_API, body=body)
    did = resp.get("domainId")
    print(f"  ✓ Created domain '{tag_key}' → {did}")
    return did


# ---- Parent domain ----
domain_id = create_or_get_domain(
    tag_key=PARENT_TAG,
    subtitle="Sourcing risk, supplier management & PO analytics",
    description=(
        "SAP Purchasing data for an automotive OEM. Contains 5 gold tables, "
        "4 metric views, 7 certified SQL functions, 2 Genie spaces, and a "
        "Multi-Agent Supervisor — everything a purchasing analyst needs to "
        "answer sourcing-risk and supply-continuity questions in plain English."
    ),
    icon_name="SHOPPING_CART",
    icon_color="#1B5E20",
)
print(f"\nParent domain ID: {domain_id}")

# ---- Subdomain 1: Sourcing & Supplier Risk ----
sub1_id = create_or_get_domain(
    tag_key=SUB_TAG_1,
    subtitle="Multi-sourcing gaps, delivery risk, purchasing blocks & quotas",
    description=(
        "Answers questions about which materials lack multi-sourcing, which "
        "suppliers carry delivery risk, purchasing blocks with active sourcing, "
        "and quota concentration. Covers Q1–Q6, Q9–Q10 of the business question set."
    ),
    icon_name="WARNING",
    icon_color="#E65100",
    parent_domain_id=domain_id,
)

# ---- Subdomain 2: Purchase Order Analytics ----
sub2_id = create_or_get_domain(
    tag_key=SUB_TAG_2,
    subtitle="On-time delivery, price variance & orders under risk",
    description=(
        "Answers questions about on-time delivery rates, net-price variance "
        "across vendors, and purchase orders placed while a supplier had an "
        "active delivery-risk flag. Covers Q7, Q8, Q11."
    ),
    icon_name="TRUCK",
    icon_color="#1565C0",
    parent_domain_id=domain_id,
)

print(f"\nSubdomain 1 (Sourcing & Risk): {sub1_id}")
print(f"Subdomain 2 (PO Analytics):     {sub2_id}")

## 3) Assign assets to domains via governed tags
Apply the governed tag to each UC object (table, function, metric view) so it
appears under the correct domain/subdomain on the Discover page.

**Shared assets** (MATERIAL, SUPPLIER, VALUE_DICTIONARY) get the **parent** tag.
**Domain-specific assets** get the **subdomain** tag.

In [0]:
def set_tag(object_type: str, fq_name: str, tag_key: str):
    """Apply a governed tag to a UC object via SQL."""
    try:
        spark.sql(f"ALTER {object_type} {fq_name} SET TAGS ('{tag_key}' = '')")
        print(f"  ✓ Tagged {object_type} {fq_name.split('.')[-1]} → {tag_key}")
    except Exception as e:
        print(f"  • Tag {fq_name.split('.')[-1]}: {e}")


# ---- Shared assets → parent domain tag ----
print("\n--- Shared assets (parent domain) ---")
for tbl in ["MATERIAL", "SUPPLIER", "VALUE_DICTIONARY"]:
    set_tag("TABLE", CFG.fq(tbl), PARENT_TAG)

# ---- Subdomain 1: Sourcing & Supplier Risk ----
print("\n--- Subdomain 1: Sourcing & Supplier Risk ---")
for tbl in ["SOURCING", "SUPPLIER_RISK"]:
    set_tag("TABLE", CFG.fq(tbl), SUB_TAG_1)

# Metric views
for mv in ["mv_sourcing_coverage", "mv_supplier_risk"]:
    set_tag("TABLE", CFG.fq(mv), SUB_TAG_1)   # MVs are TABLE objects in UC

# SQL functions (the 7 we keep)
for fn in ["materials_without_multisourcing",
           "suppliers_with_delivery_risk",
           "materials_linked_to_risky_supplier",
           "single_sourced_risky",
           "blocked_suppliers_with_active_sourcing",
           "quota_concentration",
           # no-arg wrappers used by Genie
           "materials_without_multisourcing_default",
           "materials_linked_to_risky_supplier_today",
           "single_sourced_risky_today",
           "suppliers_with_delivery_risk_today"]:
    set_tag("FUNCTION", CFG.fq(fn), SUB_TAG_1)

# ---- Subdomain 2: Purchase Order Analytics ----
print("\n--- Subdomain 2: Purchase Order Analytics ---")
set_tag("TABLE", CFG.fq("PURCHASE_ORDER"), SUB_TAG_2)

for mv in ["mv_delivery_performance", "mv_price_analytics"]:
    set_tag("TABLE", CFG.fq(mv), SUB_TAG_2)

for fn in ["pos_under_active_risk",
           "otd_rate_per_supplier_last_12m"]:
    set_tag("FUNCTION", CFG.fq(fn), SUB_TAG_2)

print("\n✓ All assets tagged.")

## 4) Create governed Pages (business glossary)
Pages are **business-term definitions** that Genie One reads as part of its
ontology. Each Page is attached to a domain or subdomain.

The 5 Pages follow the separation principle:
* **Comments** = what a column/table IS (physical)
* **Pages** = what a business concept MEANS (definitional) + WHY it matters
* **Metric Views** = HOW a KPI is calculated
* **Functions** = complex multi-table trusted queries

In [0]:
# ---- Discover the correct pages API path ----
# Pages API also lives outside /unity-catalog/.
PAGES_API = None
for candidate in ["/api/2.0/pages", "/api/2.1/pages",
                  "/api/2.0/unity-catalog/pages", "/api/2.1/unity-catalog/pages"]:
    try:
        _do("GET", candidate)
        PAGES_API = candidate
        break
    except Exception as e:
        if "Not Found" not in str(e) and "No API found" not in str(e):
            PAGES_API = candidate
            break

if PAGES_API:
    print(f"Pages API found: {PAGES_API}")
else:
    # Pages API GET might not exist but POST might — try creating to discover
    print("Pages API not found via GET — will attempt POST when creating pages.")
    PAGES_API = "/api/2.0/pages"  # Best guess for POST


def create_page(domain_id: str, name: str, description: str, body: str,
                 synonyms: list = None):
    """Create a governed Page attached to a domain/subdomain."""
    if not PAGES_API or not domain_id:
        print(f"  • Skipped Page (API or domain unavailable): {name}")
        return None
    payload = {
        "domain_id": domain_id,
        "name": name,
        "description": description,
        "body": body,
    }
    if synonyms:
        payload["synonyms"] = synonyms
    try:
        resp = _do("POST", PAGES_API, body=payload)
        pid = resp.get("id", "ok")
        print(f"  ✓ Created Page: {name} → {pid}")
        return pid
    except Exception as e:
        if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
            print(f"  • Page already exists: {name}")
            return None
        else:
            raise


# ---- Page 1: Multi-Sourcing Strategy (Subdomain 1) ----
create_page(
    domain_id=sub1_id,
    name="Multi-Sourcing Strategy",
    description="What multi-sourcing means and why it matters for supply continuity.",
    synonyms=["multi-sourcing", "single-sourced", "dual sourcing", "second source"],
    body=(
        "## Definition\n\n"
        "A material is **multi-sourced** when it has **≥2 valid, non-blocked vendors** "
        "(IS_VALID_SOURCE = true) in its source list at the relevant plant. "
        "Materials below this threshold are flagged as **single-source risks**.\n\n"
        "## Why it matters\n\n"
        "Single-sourced materials create supply-chain concentration risk: if the sole "
        "vendor experiences disruption, the production line has no fallback. "
        "Multi-sourcing is the primary mitigation.\n\n"
        "## Threshold\n\n"
        "The standard threshold is **2** (i.e. a material needs at least 2 valid vendors "
        "to be considered multi-sourced). The certified function "
        "`materials_without_multisourcing(min_vendors)` is parameterized so the threshold "
        "can be adjusted.\n\n"
        "## Related assets\n\n"
        "* **Metric view:** `mv_sourcing_coverage` — vendor count by material, plant, group\n"
        "* **Function:** `materials_without_multisourcing(min_vendors)` — Q1\n"
        "* **Table:** `SOURCING` (IS_VALID_SOURCE column)"
    ),
)

# ---- Page 2: Delivery Risk (Subdomain 1) ----
create_page(
    domain_id=sub1_id,
    name="Delivery Risk",
    description="How delivery risk is defined, assessed, and why it is time-sliced.",
    synonyms=["delivery risk", "supplier risk", "risk flag", "risk score", "risk window"],
    body=(
        "## Definition\n\n"
        "Delivery risk is a **time-sliced** assessment. A supplier carries **active** "
        "delivery risk when:\n\n"
        "1. `DELIVERY_RISK_FLAG = 'X'`, **and**\n"
        "2. The reference date falls **between VALID_FROM and VALID_TO**.\n\n"
        "Risk outside this window is historical, not current.\n\n"
        "## Risk dimensions\n\n"
        "Each risk record carries:\n"
        "* **RISK_CATEGORY** — financial, quality, geopolitical, or logistics\n"
        "* **RISK_SCORE** — numeric 0–100 (higher = more severe)\n"
        "* **DELIVERY_RISK_FLAG** — 'X' if the supplier is flagged\n\n"
        "## Why it matters\n\n"
        "A material that is **single-sourced AND its sole supplier has active delivery "
        "risk** is the highest-priority supply-continuity exposure (the '$2M question').\n\n"
        "## Related assets\n\n"
        "* **Metric view:** `mv_supplier_risk` — risk exposure by category/flag/country\n"
        "* **Function:** `single_sourced_risky(as_of_date)` — Q3\n"
        "* **Table:** `SUPPLIER_RISK` (VALID_FROM, VALID_TO columns)"
    ),
)

# ---- Page 3: On-Time Delivery (Subdomain 2) ----
create_page(
    domain_id=sub2_id,
    name="On-Time Delivery (OTD)",
    description="How OTD rate is calculated and what counts as on-time.",
    synonyms=["OTD", "on-time delivery", "on-time rate", "delivery performance"],
    body=(
        "## Definition\n\n"
        "**OTD rate** = on-time lines ÷ received lines.\n\n"
        "A PO line is **on-time** when `GR_DATE ≤ REQUESTED_DATE` (goods received by "
        "the requested date). Lines where `GR_DATE IS NULL` (not yet received) are "
        "**excluded** from both numerator and denominator.\n\n"
        "## Standard window\n\n"
        "The default analysis window is the **last 12 months** of goods-receipt dates.\n\n"
        "## Why it matters\n\n"
        "OTD is the primary measure of supplier delivery performance. Suppliers with "
        "persistently low OTD rates are candidates for escalation, re-sourcing, or "
        "additional risk mitigation.\n\n"
        "## Related assets\n\n"
        "* **Metric view:** `mv_delivery_performance` — OTD % by supplier/plant/month\n"
        "* **Table:** `PURCHASE_ORDER` (ON_TIME_FLAG, GR_DATE columns)"
    ),
)

# ---- Page 4: Quota Arrangement (Subdomain 1) ----
create_page(
    domain_id=sub1_id,
    name="Quota Arrangement",
    description="How procurement volume is split across approved vendors.",
    synonyms=["quota", "quota split", "quota concentration", "vendor share"],
    body=(
        "## Definition\n\n"
        "A **quota arrangement** splits procurement of a material across approved "
        "vendors by percentage. The `QUOTA` column in the SOURCING table represents "
        "the vendor's allocated share.\n\n"
        "## NULL quota\n\n"
        "A `NULL` quota means the vendor is **listed as a valid source** but has "
        "**no formal quota split** assigned. This is a meaningful gap, not a data "
        "error — it indicates the source list is approved but procurement volume "
        "allocation is not governed.\n\n"
        "## Concentration risk\n\n"
        "When a single vendor holds **>80%** of the quota for a material, it represents "
        "a concentration risk even if the material is technically multi-sourced. "
        "The function `quota_concentration(threshold_pct)` flags these cases.\n\n"
        "## Related assets\n\n"
        "* **Function:** `quota_concentration(threshold_pct)` — Q6\n"
        "* **Table:** `SOURCING` (QUOTA column)"
    ),
)

# ---- Page 5: SAP Code Glossary (Parent domain) ----
create_page(
    domain_id=domain_id,
    name="SAP Code Glossary",
    description="How cryptic SAP codes are translated to human-readable labels.",
    synonyms=["SAP codes", "value dictionary", "code mapping", "MTART", "SPERM", "NOTKZ"],
    body=(
        "## What it is\n\n"
        "SAP uses **short alphanumeric codes** for business concepts. The "
        "`VALUE_DICTIONARY` table maps each code to a human-readable label so that "
        "Genie (and humans) can translate between plain English and SAP terminology.\n\n"
        "## Key domains\n\n"
        "| Code domain | Example code | Meaning |\n"
        "| --- | --- | --- |\n"
        "| MTART (material type) | ROH | Raw material |\n"
        "| MATKL (material group) | 001 | Mechanical components |\n"
        "| SPERM (purchasing block) | X | Supplier is blocked for purchasing |\n"
        "| NOTKZ (source-list block) | X | Source record is blocked |\n"
        "| DELIVERY_RISK_FLAG | X | Active delivery risk |\n"
        "| LAND1 (country) | DE | Germany |\n\n"
        "## How Genie uses it\n\n"
        "When a user asks about 'blocked suppliers', Genie translates this to "
        "`SPERM = 'X'` using the value dictionary. When a user asks about 'raw "
        "materials', Genie translates to `MTART = 'ROH'`.\n\n"
        "## Related assets\n\n"
        "* **Table:** `VALUE_DICTIONARY` (DOMAIN, CODE, LABEL, DESCRIPTION columns)"
    ),
)

print("\n✓ All 5 Pages created.")

## 5) Publish the domain and subdomains
Domains are created in **draft** state (visible only to curators). Publishing
makes them visible to all users on the Discover page.

In [0]:
def publish_domain(domain_id: str, tag_key: str):
    """Publish a domain/subdomain so it is visible to consumers."""
    if not domain_id or not DOMAINS_API:
        print(f"  • Skipped publish (domain unavailable): {tag_key}")
        return
    try:
        _do("POST", f"{DOMAINS_API}/{domain_id}/publish")
        print(f"  ✓ Published: {tag_key}")
    except Exception as e:
        if "already published" in str(e).lower():
            print(f"  • Already published: {tag_key}")
        else:
            print(f"  • Publish {tag_key}: {e}")


publish_domain(domain_id, PARENT_TAG)
publish_domain(sub1_id, SUB_TAG_1)
publish_domain(sub2_id, SUB_TAG_2)

print(f"\n✓ Domain publish step complete.")

## 6) Persist domain IDs for teardown

In [0]:
from pyspark.sql import Row

spark.createDataFrame([
    Row(domain_key=PARENT_TAG, domain_id=str(domain_id), domain_type="domain"),
    Row(domain_key=SUB_TAG_1, domain_id=str(sub1_id), domain_type="subdomain"),
    Row(domain_key=SUB_TAG_2, domain_id=str(sub2_id), domain_type="subdomain"),
]).write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
 .saveAsTable(CFG.fq("_DOMAINS"))

print("Domain IDs saved to _DOMAINS:")
display(spark.table(CFG.fq("_DOMAINS")))

domains_ok = DOMAINS_API is not None
tags_ok = GOVERNED_TAGS_OK
result_parts = []
if tags_ok:
    result_parts.append("governed-tags=OK")
else:
    result_parts.append("governed-tags=SKIPPED(account-limit)")
result_parts.append("asset-tagging=OK")
if domains_ok and domain_id:
    result_parts.append(f"domains={domain_id}")
elif not tags_ok:
    result_parts.append("domains=SKIPPED(governed-tags-unavailable)")
else:
    result_parts.append("domains=SKIPPED(preview-not-enabled)")

print(f"\nOpen Discover: {w.config.host}/discover")
dbutils.notebook.exit(f"09a_create_domains: SUCCESS — {', '.join(result_parts)}")